# Tópicos Avançados em IA — Aula 02
## Previsão de séries temporais

Prof. James Moraes de Almeida

Roteiro:
1. Separar treino e teste **respeitando o tempo**.
2. Verificar se a série é estacionária (teste ADF) e diferenciar.
3. Ler a autocorrelação (ACF) e a autocorrelação parcial (PACF) para escolher o modelo.
4. Construir linhas de base (previsões ingênuas).
5. ARIMA e SARIMA com validação *walk-forward*.
6. Comparar tudo com MAE, RMSE e MASE.
7. Escolher o modelo sem olhar o teste: AIC e diagnóstico dos resíduos.
8. Para ir além — e a **Atividade ST** (no fim do notebook).

Ambiente: `conda activate topicos-ia` (arquivo `environment.yml` na pasta da disciplina).

In [ ]:
import warnings

import matplotlib.pyplot as plt
import numpy as np
import pandas as pd
from sklearn.model_selection import TimeSeriesSplit
from statsmodels.graphics.tsaplots import plot_acf, plot_pacf
from statsmodels.tsa.arima.model import ARIMA
from statsmodels.tsa.statespace.sarimax import SARIMAX
from statsmodels.stats.diagnostic import acorr_ljungbox
from statsmodels.tsa.arima_process import ArmaProcess
from statsmodels.tsa.stattools import acf, adfuller, pacf

# O statsmodels avisa a cada ajuste quando o chute inicial dos parâmetros não
# serve e ele recomeça do zero. Num laço com dezenas de ajustes isso inunda a
# tela e esconde o que importa. Silenciamos só esses avisos.
warnings.filterwarnings("ignore", module="statsmodels")
warnings.filterwarnings("ignore", message="adfuller currently returns")  # statsmodels 0.15

plt.rcParams["figure.figsize"] = (12, 4)

## 1. Os dados

Uma série sintética com os três componentes da Aula 01: tendência linear,
sazonalidade **semanal** (período de 7 dias) e ruído. Fixamos a semente para que
todos na sala vejam os mesmos números.

In [ ]:
rng = np.random.default_rng(42)
N = 200
t = np.arange(N)
tendencia = 100 + 0.8 * t
sazonalidade = 15 * np.sin(2 * np.pi * t / 7)
ruido = rng.normal(0, 4, N)

serie = pd.Series(tendencia + sazonalidade + ruido,
                  index=pd.date_range("2026-01-01", periods=N, freq="D"),
                  name="valor")
serie.plot(title="Série sintética: tendência + sazonalidade de 7 dias + ruído");

**Discussão.** O gráfico mostra os três componentes que construímos. A tendência sobe 0,8 por dia:
de 100 no primeiro dia a 259 no último. Em torno dela, uma oscilação de ±15 que se repete a cada
7 dias. O ruído, com desvio padrão 4, é a irregularidade de um dia para o outro.

Como fomos nós que construímos a série, **sabemos a resposta certa**. Ao longo do notebook vamos
conferir se cada ferramenta a encontra: o ADF, a ACF, os modelos e as métricas. Numa série real
essa conferência não existe — por isso vale treinar aqui.

## 2. Separando treino e teste

Numa série temporal **não podemos sortear** os pontos de teste: o modelo veria o
futuro durante o treino. O teste é sempre um pedaço do fim da série.

Abaixo estão três estratégias. As duas últimas servem para validação cruzada.

In [ ]:
def divisao_simples(dados, fracao_teste=0.2, plot=True):
    """Um único corte no tempo: o começo treina, o fim testa."""
    corte = int(len(dados) * (1 - fracao_teste))
    treino, teste = dados.iloc[:corte], dados.iloc[corte:]
    if plot:
        plt.figure()
        plt.plot(treino, color="royalblue", label="treino")
        plt.plot(teste, color="darkorange", label="teste")
        plt.axvline(dados.index[corte], color="red", ls="--", label="corte")
        plt.title("Divisão simples no tempo")
        plt.legend(); plt.grid(alpha=0.4)
    return treino, teste


def janela_deslizante(dados, n_janelas=5, plot=True):
    """Janelas de mesmo tamanho que andam pela série, sem sobreposição."""
    tam = len(dados) // (2 * n_janelas)
    dobras = []
    for i in range(n_janelas):
        ini = i * 2 * tam
        dobras.append((dados.iloc[ini:ini + tam], dados.iloc[ini + tam:ini + 2 * tam]))
    if plot:
        fig, eixos = plt.subplots(n_janelas, 1, figsize=(12, 1.8 * n_janelas),
                                  sharex=True, sharey=True)
        fig.suptitle("Janela deslizante")
        for i, (tr, te) in enumerate(dobras):
            eixos[i].plot(dados, color="gray", alpha=0.3)
            eixos[i].plot(tr, color="royalblue"); eixos[i].plot(te, color="darkorange")
            eixos[i].set_title(f"dobra {i + 1}", loc="left", fontsize=9)
        plt.tight_layout()
    return dobras


def janela_expansiva(dados, n_dobras=5, plot=True):
    """O treino cresce a cada dobra — é o que o TimeSeriesSplit do scikit-learn faz."""
    dobras = [(dados.iloc[a], dados.iloc[b])
              for a, b in TimeSeriesSplit(n_splits=n_dobras).split(dados)]
    if plot:
        fig, eixos = plt.subplots(n_dobras, 1, figsize=(12, 1.8 * n_dobras),
                                  sharex=True, sharey=True)
        fig.suptitle("Janela em expansão (TimeSeriesSplit)")
        for i, (tr, te) in enumerate(dobras):
            eixos[i].plot(dados, color="gray", alpha=0.3)
            eixos[i].plot(tr, color="royalblue"); eixos[i].plot(te, color="darkorange")
            eixos[i].set_title(f"dobra {i + 1}", loc="left", fontsize=9)
        plt.tight_layout()
    return dobras

In [ ]:
treino, teste = divisao_simples(serie)
dobras_desl = janela_deslizante(serie)
dobras_exp = janela_expansiva(serie)
print(f"divisão simples: treino {len(treino)} dias ({treino.index[0]:%d/%m} a {treino.index[-1]:%d/%m}), "
      f"teste {len(teste)} dias ({teste.index[0]:%d/%m} a {teste.index[-1]:%d/%m})")
print("janela deslizante, (treino, teste) em dias:", [(len(a), len(b)) for a, b in dobras_desl])
print("janela em expansão, (treino, teste) em dias:", [(len(a), len(b)) for a, b in dobras_exp])

**Pergunta:** na janela deslizante, o modelo da dobra 5 nunca vê o começo da série.
Quando isso é uma vantagem, e quando é um problema?

**Discussão.** Os tamanhos impressos acima:
* **divisão simples:** 160 dias de treino (01/01 a 09/06) e 40 de teste (10/06 a 19/07). É a divisão
  que usamos no resto da aula.
* **janela deslizante:** 5 dobras de 20 dias de treino e 20 de teste. Vinte dias são menos de três
  semanas — pouco para estimar uma sazonalidade semanal. Um SARIMA com as duas diferenças ainda
  perde 8 desses 20 pontos só para diferenciar.
* **janela em expansão:** o teste tem sempre 33 dias e o treino cresce de 35 para 167. Cada dobra
  usa todo o passado disponível.

Sobre a pergunta: esquecer o começo da série é **vantagem** quando o processo muda com o tempo — um
equipamento trocado, um regime novo de operação, clientes que mudaram de hábito. Os dados antigos
descrevem um sistema que já não existe e atrapalham. É **problema** quando a série é estável e curta,
como a nossa: jogar dados fora só aumenta a incerteza dos parâmetros. Aqui a regra que gera os dados
não muda nos 200 dias, então a janela em expansão é a escolha natural.

### Por que não sortear: a teoria

Numa tabela comum supomos as linhas **independentes e identicamente distribuídas**. Numa série,
$y_t$ e $y_{t-1}$ são correlacionados — é exatamente o que os modelos desta aula exploram. Se o
dia 10 cai no teste e os dias 9 e 11 no treino, o modelo "adivinha" o dia 10 olhando os vizinhos,
e o erro medido sai otimista: ele nunca teria esses vizinhos no momento real da previsão.

Duas palavras que usamos daqui em diante:
* **origem da previsão** $t$: o último instante conhecido;
* **horizonte** $h$: quantos passos à frente se prevê. A previsão se escreve $\hat y_{t+h\mid t}$
  — "o valor em $t+h$, sabendo até $t$".

A validação cruzada em séries (janelas deslizantes ou em expansão) move a origem pela série e
mede o erro em várias origens — o que dá uma ideia de quanto o erro varia, e não só um número.

## 3. A série é estacionária?

### Estacionariedade

Uma série é **(fracamente) estacionária** quando as suas propriedades estatísticas não dependem
do instante:

$$\mathbb{E}[y_t] = \mu, \qquad \operatorname{Var}(y_t) = \sigma^2, \qquad
\operatorname{Cov}(y_t,\, y_{t-k}) = \gamma_k \quad \text{para todo } t .$$

A covariância pode depender da **distância** $k$ entre os pontos, mas não de **onde** eles estão.
Três exemplos:

* **ruído branco**, $y_t = \varepsilon_t$ com $\varepsilon_t$ independentes de média 0: estacionário;
* **tendência**, $y_t = a + bt + \varepsilon_t$: a média $a + bt$ muda com $t$ — não estacionária;
* **passeio aleatório**, $y_t = y_{t-1} + \varepsilon_t$: a média é constante, mas
  $\operatorname{Var}(y_t) = t\,\sigma^2$ cresce sem limite — não estacionário.

Por que isso importa: ARMA, a base do ARIMA, só descreve séries estacionárias. Ele aprende
"como $y_t$ se relaciona com o seu passado", e essa relação precisa ser a mesma ao longo da série.

### Diferenciar

O **operador de defasagem** $L$ recua um passo: $L\,y_t = y_{t-1}$, e $L^k y_t = y_{t-k}$. A
**diferença** é $\Delta y_t = (1 - L)\,y_t = y_t - y_{t-1}$, e a diferença sazonal de período $m$ é
$(1 - L^m)\,y_t = y_t - y_{t-m}$.

* No passeio aleatório, $(1-L)\,y_t = \varepsilon_t$: uma diferença o torna ruído branco.
* Numa tendência linear, $(1-L)(a + bt) = b$: a diferença vira uma constante.
* Na sazonalidade de período $m$, $(1-L^m)\,c_t = 0$: a diferença sazonal a elimina.

O `d` do ARIMA é quantas vezes aplicamos $(1-L)$; o `D` é quantas vezes aplicamos $(1-L^m)$.

### O teste de Dickey-Fuller aumentado (ADF)

O ADF ajusta a regressão

$$\Delta y_t = \alpha + \gamma\, y_{t-1} + \sum_{i=1}^{p} \delta_i\, \Delta y_{t-i} + \varepsilon_t$$

e testa $H_0\!: \gamma = 0$ — a **raiz unitária**, o comportamento de passeio aleatório — contra
$\gamma < 0$, em que a série volta para a média. Os termos $\Delta y_{t-i}$ ("aumentado") absorvem
a autocorrelação de curto prazo; o `adfuller` escolhe quantos pelo AIC.

Um detalhe que engana: a estatística é a razão $t$ de $\hat\gamma$, mas **sob $H_0$ ela não segue a
distribuição $t$ de Student**. Dickey e Fuller (1979) tabelaram a distribuição certa; o
`statsmodels` usa os valores críticos de MacKinnon.

* p-valor **grande** (> 0,05): não dá para rejeitar a raiz unitária — trate como não estacionária.
* p-valor **pequeno**: rejeita-se a raiz unitária.

Não rejeitar $H_0$ **não prova** que a série tem raiz unitária — com poucos dados o teste tem
pouco poder.

In [ ]:
def adf(x, nome):
    estat, p, _, _, criticos, *_ = adfuller(x.dropna())
    print(f"{nome:32s} estatística = {estat:6.2f}   crítico 5% = {criticos['5%']:5.2f}   p-valor = {p:.4f}")

adf(serie, "original")
adf(serie.diff(), "1 diferença (d = 1)")
adf(serie.diff().diff(7), "1 diferença + 1 sazonal (D = 1)")

**Discussão.** Para rejeitar a raiz unitária com 5% de significância, a estatística precisa ficar
**abaixo** do valor crítico, −2,88.
* **Série original:** −0,02, longe do crítico; p = 0,96. Não dá para rejeitar a raiz unitária: para
  este teste, a série se comporta como um passeio aleatório. (Logo adiante veremos que é a tendência
  que o confunde.)
* **Uma diferença (`d = 1`):** −9,01, muito abaixo do crítico. O p-valor impresso como 0,0000 quer
  dizer menor que 0,00005. A tendência virou uma constante (0,8) e a raiz unitária foi rejeitada.
* **Mais a diferença sazonal (`D = 1`):** −7,61 — continua rejeitando.

Repare que **o ADF não pede o `D`**: com `d = 1` ele já passa, embora a oscilação semanal continue lá.
O teste procura raiz unitária, não sazonalidade. Quem pede a diferença sazonal é a ACF, na seção 4,
com o pico no lag 7.

### Conferindo o teste onde sabemos a resposta

Um ruído branco e um passeio aleatório construídos do mesmo ruído:

In [ ]:
g = np.random.default_rng(7)                      # semente própria: não mexe na série da aula
choque = g.normal(0, 1, 300)
passeio = pd.Series(np.cumsum(choque))
adf(pd.Series(choque), "ruído branco")
adf(passeio, "passeio aleatório")
adf(passeio.diff(), "passeio aleatório, 1 diferença")

**Discussão.** O teste acerta os dois casos em que sabemos a resposta:
* **ruído branco:** −17,38, muito abaixo do crítico — rejeita a raiz unitária. Correto: o ruído é
  estacionário.
* **passeio aleatório:** −1,69, acima do crítico, p = 0,44 — não rejeita. Correto: o passeio tem raiz
  unitária.
* **passeio diferenciado:** −17,36, praticamente o número do ruído branco. Não é coincidência:
  $(1-L)$ aplicado ao passeio devolve exatamente os choques que o construíram, menos o primeiro.

Mas **não rejeitar não é provar**. Um AR(1) com $\phi = 0{,}98$ é estacionário; com 300 pontos, o ADF
só rejeita a raiz unitária em cerca de 1 de cada 5 séries assim (conferido por simulação). Perto de
$\phi = 1$ o teste tem pouco poder. Por isso se usa o KPSS junto quando a dúvida importa (seção 9).

### Tendência determinística ou raiz unitária?

O `adfuller` padrão (`regression="c"`) só tem a constante $\alpha$ na regressão. Uma série com
tendência **determinística** $a + bt$ e ruído estacionário — como a nossa, que construímos assim —
também "não passa" nesse teste, porque a regressão não tem como explicar a tendência. Com
`regression="ct"` entra um termo $\beta t$, e o teste passa a perguntar: *tirada a tendência, sobra
algo estacionário?*

Na série sintética podemos tirar a sazonalidade verdadeira e perguntar só pela tendência:

In [ ]:
def adf_ct(x, nome):
    estat, p, *_ = adfuller(x.dropna(), regression="ct")
    print(f"{nome:44s} p-valor = {p:.4g}")

adf_ct(serie - sazonalidade, "série da aula sem a sazonalidade, com tendência")
adf_ct(passeio, "passeio aleatório, com tendência")

**Discussão.** Com o termo de tendência na regressão, o teste separa as duas séries:
* **a nossa, sem a sazonalidade:** p = 2,2 × 10⁻¹³ — rejeita a raiz unitária com folga. Tirada a
  reta, sobra algo estacionário, exatamente como a construímos: reta mais ruído independente.
* **o passeio aleatório:** p = 0,28 — continua sem rejeitar. Nenhuma reta explica um passeio: a raiz
  unitária é de verdade.

Na primeira célula da seção, a série original "parecia" ter raiz unitária (p = 0,96) só porque o teste
sem `"ct"` não tinha como explicar a tendência.

**As duas séries sobem, mas por mecanismos diferentes.** A nossa é *estacionária em torno de uma
tendência*: um choque de hoje some amanhã. O passeio aleatório tem *raiz unitária*: todo choque
fica para sempre. Diferenciar funciona para as duas na prática — mas na nossa ele "sobrediferencia",
e isso vai aparecer nos parâmetros estimados (seção 4).

## 4. ACF e PACF: escolhendo `p` e `q`

### Os dois modelos de base

**Autorregressivo de ordem $p$, AR($p$):** o valor de hoje é uma combinação dos $p$ valores anteriores,

$$y_t = c + \phi_1 y_{t-1} + \dots + \phi_p y_{t-p} + \varepsilon_t .$$

**Média móvel de ordem $q$, MA($q$):** o valor de hoje é uma combinação dos $q$ **erros** anteriores,

$$y_t = \mu + \varepsilon_t + \theta_1 \varepsilon_{t-1} + \dots + \theta_q \varepsilon_{t-q} .$$

Cuidado com o nome: a "média móvel" do MA **não** é a média móvel de suavização da Aula 01. É uma
regressão nos choques passados — por isso, nos slides, `q` é "o número de erros passados no modelo".

Com o operador $L$: $\phi(L)\,y_t = \theta(L)\,\varepsilon_t$, com $\phi(L) = 1 - \phi_1 L - \dots - \phi_p L^p$ e
$\theta(L) = 1 + \theta_1 L + \dots + \theta_q L^q$. O **ARIMA($p,d,q$)** é um ARMA aplicado à série
diferenciada: $\phi(L)\,(1-L)^d\, y_t = \theta(L)\,\varepsilon_t$.

Duas condições técnicas: o AR(1) só é estacionário com $|\phi_1| < 1$; o MA(1) só é **invertível**
(dá para recuperar os choques a partir dos dados) com $|\theta_1| < 1$. É desta última que fala o
aviso "non-invertible starting MA parameters" que o `statsmodels` solta durante o ajuste.

### ACF e PACF

* **ACF**, autocorrelação no lag $k$: $\rho_k = \gamma_k / \gamma_0$ — a correlação entre $y_t$ e $y_{t-k}$.
* **PACF**, autocorrelação **parcial** no lag $k$: a correlação entre $y_t$ e $y_{t-k}$ **descontado o
  efeito dos lags intermediários** $1, \dots, k-1$. É o último coeficiente de uma regressão AR($k$).

As assinaturas teóricas, que é o que se procura nos gráficos:

| modelo | ACF | PACF |
|---|---|---|
| AR($p$) | decai aos poucos (no AR(1), $\rho_k = \phi^k$) | **corta** depois do lag $p$ |
| MA($q$) | **corta** depois do lag $q$ (no MA(1), $\rho_1 = \theta/(1+\theta^2)$) | decai aos poucos |

A PACF corta no AR porque, conhecidos os $p$ últimos valores, os anteriores não acrescentam nada.
A ACF corta no MA porque $y_t$ e $y_{t-k}$ não têm nenhum choque em comum quando $k > q$.

**A faixa azul.** Para ruído branco, $\hat\rho_k$ tem desvio padrão de cerca de $1/\sqrt{n}$, e a faixa
de 95% é $\pm 1{,}96/\sqrt{n}$. No gráfico da ACF o `statsmodels` usa a fórmula de Bartlett, que
alarga a faixa a cada lag já significativo — por isso ela cresce para a direita.

### Conferindo as assinaturas em séries simuladas

In [ ]:
g = np.random.default_rng(1)
phi, theta = 0.7, 0.7
ar1 = ArmaProcess(ar=[1, -phi], ma=[1]).generate_sample(500, distrvs=g.standard_normal)
ma1 = ArmaProcess(ar=[1], ma=[1, theta]).generate_sample(500, distrvs=g.standard_normal)

fig, eixos = plt.subplots(2, 2, figsize=(12, 5.5))
for i, (x, nome) in enumerate([(ar1, f"AR(1), φ = {phi}"), (ma1, f"MA(1), θ = {theta}")]):
    plot_acf(x, lags=12, ax=eixos[i, 0], title=f"ACF — {nome}")
    plot_pacf(x, lags=12, ax=eixos[i, 1], method="ywm", title=f"PACF — {nome}")
plt.tight_layout()

print(f"faixa de 95% da PACF: ±{1.96 / np.sqrt(len(ar1)):.3f}   (a da ACF alarga com o lag)")
for x, proc, nome in [(ar1, ArmaProcess(ar=[1, -phi], ma=[1]), "AR(1)"),
                      (ma1, ArmaProcess(ar=[1], ma=[1, theta]), "MA(1)")]:
    print()
    print(f"{nome}           lag:" + "".join(f"{k:7d}" for k in range(1, 6)))
    print("  ACF  amostral     " + "".join(f"{v:7.2f}" for v in acf(x, nlags=5)[1:]))
    print("  ACF  teórica      " + "".join(f"{v:7.2f}" for v in proc.acf(6)[1:]))
    print("  PACF amostral     " + "".join(f"{v:7.2f}" for v in pacf(x, nlags=5, method="ywm")[1:]))
    print("  PACF teórica      " + "".join(f"{v:7.2f}" for v in proc.pacf(6)[1:]))

**Discussão.** Os dois processos foram feitos com o mesmo coeficiente, 0,7, e os gráficos saem
**espelhados**: o que corta num, decai no outro.

**AR(1), linha de cima.**
* A **ACF decai aos poucos**: 0,62, 0,41, 0,32, 0,23, 0,14… até sumir dentro da faixa. Cada passo
  multiplica pelo mesmo fator, como em $\rho_k = \phi^k$. A memória passa em cadeia: $y_t$ depende de
  $y_{t-1}$, que depende de $y_{t-2}$, e assim por diante — então $y_t$ ainda "lembra" de $y_{t-5}$.
* A **PACF corta**: uma barra só, 0,62 no lag 1, e todas as outras dentro da faixa. Descontado
  $y_{t-1}$, os lags mais antigos não acrescentam nada, porque toda a cadeia passa por ele. Leitura:
  **p = 1**.

**MA(1), linha de baixo.**
* A **ACF corta**: 0,50 no lag 1 e nada depois. $y_t$ e $y_{t-1}$ têm o choque $\varepsilon_{t-1}$ em
  comum; $y_t$ e $y_{t-2}$ não têm nenhum. Leitura: **q = 1**.
* A **PACF decai alternando o sinal**: 0,50, −0,28, 0,16, −0,16, 0,12. O motivo: um MA(1) invertível
  é um AR de ordem infinita, $y_t = \varepsilon_t + \theta y_{t-1} - \theta^2 y_{t-2} + \theta^3 y_{t-3} - \dots$,
  com coeficientes que alternam e diminuem. A PACF, "o último coeficiente do AR($k$)", herda esse
  padrão e nunca corta.

**Amostral não é teórico.** No AR(1), $\hat\rho_1 = 0{,}62$ contra 0,70 da teoria; com 500 pontos a
estimativa flutua. Troque a semente (`default_rng(2)`, `(3)`…) e os números mudam, mas a **forma** —
decai ou corta — fica. É a forma que se lê. Pelo mesmo motivo, uma barra isolada perto da borda da
faixa não é sinal: a faixa é de 95%, e em 12 lags é comum uma barra sair por acaso.

**Um limite útil.** No MA(1), $\rho_1 = \theta/(1+\theta^2)$ nunca passa de 0,5 (o máximo é em
$\theta = 1$). Uma ACF com uma única barra bem acima de 0,5 no lag 1 não é de um MA(1) puro.

**O que levar para a nossa série:** se a **PACF** corta, procure termos AR; se a **ACF** corta,
procure termos MA.

### Agora a nossa série

Olhamos a série **já diferenciada**. As faixas azuis são o intervalo de 95%:
barras dentro delas são indistinguíveis de zero.

| padrão | sugere |
|---|---|
| PACF corta depois do lag *k*, ACF decai aos poucos | AR(*k*) → `p = k` |
| ACF corta depois do lag *k*, PACF decai aos poucos | MA(*k*) → `q = k` |
| pico isolado no lag do período (7, 14…) | termo **sazonal** (`P` ou `Q`) |

Cuidado: o gráfico de autocorrelação da série **original** mistura tendência e sazonalidade e
não serve para escolher `p` — é preciso diferenciar antes.

In [ ]:
estacionaria = serie.diff().diff(7).dropna()
fig, eixos = plt.subplots(1, 2, figsize=(12, 3.5))
plot_acf(estacionaria, lags=21, ax=eixos[0])
plot_pacf(estacionaria, lags=21, ax=eixos[1], method="ywm")
eixos[0].set_title("ACF da série diferenciada")
eixos[1].set_title("PACF da série diferenciada")

r_, pr_ = acf(estacionaria, nlags=10), pacf(estacionaria, nlags=10, method="ywm")
print(f"{len(estacionaria)} pontos; faixa de 95% da PACF: ±{1.96 / np.sqrt(len(estacionaria)):.2f}")
print("lag    ACF   PACF")
for k in range(1, 11):
    print(f"{k:3d}  {r_[k]:5.2f}  {pr_[k]:5.2f}")

**Discussão — a leitura dos dois gráficos.**
* **ACF:** dois picos fortes e negativos, no lag 1 (−0,39) e no lag 7 (−0,47). Fora deles, o lag 3
  (−0,22), o lag 8 (+0,22) e o lag 10 (+0,20) ficam na borda da faixa; o resto fica dentro. A ACF
  **corta**.
* **PACF:** picos no lag 1 (−0,39), no lag 3 (−0,35) e no lag 7 (−0,42), e depois do 7 uma fila longa
  de barras negativas que vão diminuindo devagar. A PACF **decai**.

Pela seção anterior, ACF que corta e PACF que decai é a assinatura de **termos MA**. Os cortes da ACF
estão no lag 1 e no lag do período, 7: um MA(1) não sazonal e um MA(1) sazonal. Com as diferenças
`d = 1` e `D = 1`, o modelo é o SARIMA (p=0, d=1, q=1)(P=0, D=1, Q=1, m=7), escrito
`SARIMA(0,1,1)(0,1,1,7)` — o chamado *modelo das companhias aéreas*, que vamos reencontrar na
atividade.

Dois detalhes que valem atenção:
* **Por que os picos são negativos.** Diferenciar um ruído branco produz exatamente $\rho_1 = -0{,}5$:
  $\varepsilon_t - \varepsilon_{t-1}$ e $\varepsilon_{t-1} - \varepsilon_{t-2}$ dividem o choque
  $\varepsilon_{t-1}$ com sinais opostos. Picos negativos perto de −0,5 depois de diferenciar são um
  aviso de que a diferença pode ter sido demais — voltaremos a isso no ajuste.
* **O lag 3 e o lag 8.** O lag 8 positivo é previsto pelo próprio modelo (próxima célula). O lag 3 não
  é: é uma barra isolada, com 192 pontos e ruído. Não modelamos cada barra; o teste de Ljung-Box da
  seção 7b dirá se ficou algo importante para trás.

### O SARIMA em uma equação

O SARIMA$(p,d,q)(P,D,Q)_m$ multiplica os polinômios não sazonais pelos sazonais, que agem em $L^m$:

$$\phi(L)\,\Phi(L^m)\,(1-L)^d\,(1-L^m)^D\, y_t = \theta(L)\,\Theta(L^m)\,\varepsilon_t .$$

No `statsmodels`: `order=(p, d, q)` e `seasonal_order=(P, D, Q, m)`. O modelo das companhias aéreas
(Box e Jenkins, 1970), com $m = 7$:

$$(1-L)(1-L^7)\, y_t = (1 + \theta L)(1 + \Theta L^7)\,\varepsilon_t
= \varepsilon_t + \theta\,\varepsilon_{t-1} + \Theta\,\varepsilon_{t-7} + \theta\Theta\,\varepsilon_{t-8} .$$

O produto cria um termo em $\varepsilon_{t-8}$ — e por isso a ACF deste modelo não é zero só nos lags 1
e 7: ela também tem valores nos lags 6 e 8. Ajustamos o modelo **no treino** e comparamos a ACF que
ele prevê com a medida:

In [ ]:
ajuste = SARIMAX(treino.values, order=(0, 1, 1), seasonal_order=(0, 1, 1, 7)).fit(disp=False)
th, Th = ajuste.params[0], ajuste.params[1]
coef_ma = np.zeros(9); coef_ma[[0, 1, 7, 8]] = [1, th, Th, th * Th]
acf_teorica = ArmaProcess(ar=[1], ma=coef_ma).acf(9)
acf_medida = acf(treino.diff().diff(7).dropna(), nlags=9)
print(f"θ estimado = {th:.3f}   Θ estimado = {Th:.3f}")
print("lag   ACF do modelo   ACF medida no treino")
for k in (1, 2, 6, 7, 8):
    print(f"{k:3d}   {acf_teorica[k]:13.3f}   {acf_medida[k]:20.3f}")

**Discussão.** **O modelo reproduz a ACF medida** — inclusive os lags 6 e 8, que ninguém escolheu:
0,250 previsto contra 0,194 e 0,288 medidos. No lag 2 o modelo diz zero e a medida é 0,102. O treino
diferenciado tem 160 − 8 = 152 pontos, e a faixa de 95% é $\pm 1{,}96/\sqrt{152} \approx \pm 0{,}16$:
0,102 é compatível com zero.

E repare nos parâmetros: $\theta \approx \Theta \approx -1$. Com $\theta = -1$, o lado direito vira
$(1-L)(1-L^7)\,\varepsilon_t$ — o mesmo fator do lado esquerdo, que se cancela. É a assinatura da
**sobrediferenciação**: a série não tinha raiz unitária (seção 3), tinha tendência e sazonalidade
**fixas**, e o MA desfez as diferenças. O modelo continua prevendo bem — ele reencontrou, pelo
caminho do ARIMA, uma tendência e uma sazonalidade determinísticas. Numa série real, $\theta$ muito
perto de $-1$ é um convite a tentar a regressão com tendência e variáveis de calendário.

O valor impresso, −1,000, é a estimativa **colada no limite**: com a opção padrão
`enforce_invertibility=True`, o `statsmodels` mantém $|\theta| < 1$ (a invertibilidade da seção 4),
e o ótimo da verossimilhança ficou na borda.

## 5. Linhas de base

Antes de qualquer modelo, a pergunta é: **ele ganha de uma regra boba?**

* **ingênua:** amanhã = hoje, $\hat y_{t+1\mid t} = y_t$.
* **ingênua sazonal:** amanhã = o mesmo dia da semana passada, $\hat y_{t+1\mid t} = y_{t+1-m}$.

A ingênua é a previsão ótima de um passeio aleatório: é o ARIMA(0,1,0). A ingênua sazonal é o
ARIMA(0,0,0)(0,1,0)$_m$.

Um modelo que não ganha da ingênua sazonal não aprendeu nada útil.

## 6. Validação *walk-forward*

Prevemos **um passo** à frente, revelamos o valor verdadeiro, acrescentamos ao
histórico e prevemos o próximo. Assim cada previsão usa só o passado.

A função recebe uma regra `prever(historico) → próximo valor`, e todos os modelos
passam pelo mesmo laço.

**O que acontece dentro de cada `fit`.** Os parâmetros ($\phi$, $\theta$, $\Theta$ e a variância
$\sigma^2$ do ruído) são estimados por **máxima verossimilhança**: o `statsmodels` escreve o modelo em
forma de espaço de estados e calcula a verossimilhança com o filtro de Kalman. Reajustar a cada
passo é o mais honesto e o mais caro; a alternativa barata é ajustar uma vez e só atualizar o estado
com as observações novas (`ajuste.append(novos, refit=False)`).

In [ ]:
def walk_forward(treino, teste, prever):
    historico = list(treino.values)
    previsoes = []
    for k in range(len(teste)):
        previsoes.append(prever(np.array(historico)))
        historico.append(teste.iloc[k])      # só agora o modelo "vê" este ponto
    return pd.Series(previsoes, index=teste.index)

In [ ]:
modelos = {
    "ingênua":          lambda h: h[-1],
    "ingênua sazonal":  lambda h: h[-7],
    "ARIMA(1,1,1)":     lambda h: ARIMA(h, order=(1, 1, 1)).fit().forecast()[0],
    "ARIMA(7,1,0)":     lambda h: ARIMA(h, order=(7, 1, 0)).fit().forecast()[0],
    "SARIMA(0,1,1)(0,1,1,7)": lambda h: SARIMAX(
        h, order=(0, 1, 1), seasonal_order=(0, 1, 1, 7)).fit(disp=False).forecast()[0],
}

previsoes = {nome: walk_forward(treino, teste, f) for nome, f in modelos.items()}

In [ ]:
ax = teste.plot(color="black", lw=2, label="teste", figsize=(12, 4))
for nome in ["ingênua sazonal", "ARIMA(1,1,1)", "SARIMA(0,1,1)(0,1,1,7)"]:
    previsoes[nome].plot(ax=ax, ls="--", label=nome)
ax.legend(); ax.set_title("Previsões um passo à frente no conjunto de teste");

In [ ]:
print("erro médio no teste, real − previsto (positivo = a previsão fica abaixo):")
for nome, p in previsoes.items():
    print(f"  {nome:24s} {np.mean(teste - p):6.2f}")
print(f"a tendência sobe em 7 dias: 7 × 0,8 = {7 * 0.8:.1f}")

**Discussão — o que o gráfico mostra.**
* **Ingênua sazonal (azul):** tem o formato certo, porque repete a semana anterior, mas fica quase
  sempre **abaixo** da linha preta. O erro médio é +5,49, perto dos 5,6 que a tendência sobe em 7
  dias. A regra "igual à semana passada" não sabe que a série cresce.
* **ARIMA (p=1, d=1, q=1) (laranja):** chega atrasada às viradas e passa do ponto. Depois de um dia de
  subida ela prevê mais subida; quando a série vira, erra no sentido oposto. A memória dela é de um
  dia: o ciclo de 7 dias é invisível para o modelo. O viés é pequeno (+0,82) — o problema dela não é
  ficar abaixo, é o tempo.
* **SARIMA (verde):** acompanha subidas e descidas e quase não tem viés (+0,61). O que sobra parece
  ruído — a próxima seção mede quanto.

## 7. Métricas

Com o erro $e_t = y_t - \hat y_t$ nos $n$ pontos de teste:

$$\text{MAE} = \frac{1}{n}\sum_t |e_t|, \qquad \text{RMSE} = \sqrt{\frac{1}{n}\sum_t e_t^2}, \qquad
\text{MASE} = \frac{\text{MAE}}{\dfrac{1}{T-m}\sum_{t=m+1}^{T} |y_t - y_{t-m}|} .$$

* **MAE** — na unidade da série.
* **RMSE** — também na unidade da série, mas pune mais os erros grandes. Sempre $\text{RMSE} \ge \text{MAE}$;
  quanto maior a razão, mais o erro está concentrado em poucos pontos.
* **MASE** (Hyndman e Koehler, 2006) — o denominador é o MAE da **ingênua sazonal no treino**
  ($T$ pontos).
  * MASE < 1: melhor que repetir a semana passada.
  * Não tem unidade: dá para comparar séries diferentes. É a métrica da
    competição da Aula 03.
* E o **MAPE** (erro percentual médio)? Ele divide por $y_t$: explode quando a série passa perto de
  zero e pune mais os erros para cima que para baixo. Por isso não o usamos.

Atenção: o `mean_absolute_scaled_error` do skforecast divide pela ingênua de **um**
passo, não pela sazonal. Para séries sazonais, escrevemos a nossa.

In [ ]:
def mase(real, previsto, treino, m):
    """MASE sazonal: escala pelo erro da previsão ingênua de período m no treino."""
    escala = np.mean(np.abs(treino.values[m:] - treino.values[:-m]))
    return np.mean(np.abs(np.asarray(real) - np.asarray(previsto))) / escala


tabela = pd.DataFrame({
    nome: {"MAE": np.mean(np.abs(teste - p)),
           "RMSE": np.sqrt(np.mean((teste - p) ** 2)),
           "MASE": mase(teste, p, treino, m=7)}
    for nome, p in previsoes.items()
}).T.sort_values("MASE")
tabela.round(3)

**Perguntas para discutir** (pense antes de ler a discussão):
1. Por que o `ARIMA(1,1,1)` perde para a ingênua sazonal?
2. O `ARIMA(7,1,0)` não tem termo sazonal. Como ele consegue capturar parte da
   sazonalidade?
3. O que o `SARIMA` sabe que o `ARIMA(7,1,0)` precisa descobrir?

Antes da discussão, uma referência: **quanto é possível acertar?** Como construímos a série,
conhecemos a tendência e a sazonalidade verdadeiras. Um "oráculo" que as usa só erra o ruído:

In [ ]:
oraculo = pd.Series(tendencia[len(treino):] + sazonalidade[len(treino):], index=teste.index)
e_or = teste - oraculo
print(f"oráculo:  MAE = {np.mean(np.abs(e_or)):.3f}   RMSE = {np.sqrt(np.mean(e_or ** 2)):.3f}   "
      f"MASE = {mase(teste, oraculo, treino, m=7):.3f}")
print("razão RMSE/MAE de cada modelo:", (tabela["RMSE"] / tabela["MAE"]).round(2).to_dict())
print(f"para erros normais, RMSE/MAE = √(π/2) = {np.sqrt(np.pi / 2):.2f}")

**Discussão da tabela.**
* **A ordem:** SARIMA (MASE 0,53) < ARIMA (p=7, d=1, q=0) (0,72) < ingênua sazonal (1,02) <
  ARIMA (p=1, d=1, q=1) (1,38) < ingênua (1,50). Dois modelos ganham da ingênua sazonal; um ARIMA perde.
* **O piso:** nenhum modelo erra menos que o ruído. O oráculo tem MAE 3,07 e MASE 0,49 — abaixo disso
  é impossível nesta série. O SARIMA, com MAE 3,26, está a cerca de 6% do piso: quase tudo o que era
  previsível ele previu. Numa série real não há oráculo, mas o piso existe e não é zero.
* **A ingênua sazonal com MASE ≈ 1** não é coincidência: a MASE divide justamente pelo erro dela no
  treino. No teste ela erra o mesmo que no treino porque a série não mudou de comportamento.
* **RMSE/MAE** fica entre 1,17 e 1,26 em todos os modelos, perto de 1,25, o valor de erros normais.
  Nenhum modelo tem poucos erros enormes; os erros são espalhados. A ingênua sazonal tem a razão mais
  baixa porque o erro dela é quase sempre do mesmo sinal (o viés de +5,5).

**As três perguntas.**
1. O ARIMA (p=1, d=1, q=1) só olha um dia para trás. O ciclo de 7 dias fica inteiro nos resíduos — o
   Ljung-Box da seção 7b confirma —, e nas viradas ele extrapola a última variação. A ingênua sazonal,
   mesmo boba, tem o período certo.
2. Com 7 lags da série diferenciada, o ARIMA (p=7, d=1, q=0) enxerga o valor de uma semana atrás, e
   a regressão aprende uma combinação que reproduz boa parte do ciclo. Mas precisa estimar 7
   coeficientes com 160 pontos, sem saber que o padrão é exatamente semanal.
3. O SARIMA recebe pronto o período (m = 7) e a forma do padrão sazonal (diferença sazonal mais um
   termo MA sazonal). São 2 parâmetros contra 7. **Saber a estrutura vale dados.**

## 7b. Escolher o modelo sem olhar o teste

A tabela acima usou o teste — e numa situação real o teste é o futuro, que não temos. Duas
ferramentas usam **só o treino**.

**AIC** (critério de informação de Akaike): $\text{AIC} = 2k - 2\ln \hat{\mathcal{L}}$, com $k$ o número
de parâmetros e $\hat{\mathcal{L}}$ a verossimilhança máxima. Premia o ajuste e pune a complexidade;
menor é melhor. Só compara modelos **com a mesma diferenciação** (a verossimilhança é calculada em
séries diferentes quando $d$ ou $D$ mudam) — aqui os três têm $d = 1$, mas o SARIMA tem também
$D = 1$, então a comparação é indicativa. É o critério que o AutoARIMA da Aula 03 minimiza.

**Resíduos de um bom modelo são ruído branco.** Se sobrou autocorrelação nos resíduos, sobrou
informação que o modelo não usou. O teste de **Ljung-Box** junta as autocorrelações dos resíduos
até o lag $h$,

$$Q = n(n+2)\sum_{k=1}^{h} \frac{\hat\rho_k^2}{n-k},$$

e testa $H_0$: "não há autocorrelação". p-valor pequeno = sobrou estrutura.

In [ ]:
candidatos = {"ARIMA(1,1,1)": ((1, 1, 1), (0, 0, 0, 0)),
              "ARIMA(7,1,0)": ((7, 1, 0), (0, 0, 0, 0)),
              "SARIMA(0,1,1)(0,1,1,7)": ((0, 1, 1), (0, 1, 1, 7))}
for nome, (o, so) in candidatos.items():
    r = SARIMAX(treino.values, order=o, seasonal_order=so).fit(disp=False)
    lb = acorr_ljungbox(r.resid[8:], lags=[14])          # os primeiros resíduos são do "aquecimento"
    print(f"{nome:24s} AIC = {r.aic:7.1f}   Ljung-Box (14 lags): p = {lb['lb_pvalue'].iloc[0]:.4f}")

**Discussão.**
* **AIC:** 1153,9, 968,2 e 847,0. Menor é melhor, e diferenças acima de 10 já são consideradas fortes;
  aqui são de mais de 100. O SARIMA tem `D = 1` e por isso a comparação é indicativa, mas a ordem é
  clara.
* **Ljung-Box:** no ARIMA (p=1, d=1, q=1), p < 0,0001 — sobrou muita estrutura nos resíduos, o ciclo
  semanal inteiro. No ARIMA (p=7, d=1, q=0), p = 0,013 — sobrou um pouco. No SARIMA, p = 0,32 — os
  resíduos são compatíveis com ruído branco, e o lag 3 que deixamos de fora não fez falta.

O AIC e o Ljung-Box, calculados **só no treino**, apontam o mesmo modelo que a tabela do teste. É
isso que se quer: escolher sem olhar o futuro, e usar o teste uma vez só, para medir.

## 8. SARIMAX: variáveis exógenas

O **X** de SARIMAX é uma variável externa conhecida no instante da previsão:
temperatura, preço do dólar, um feriado. Ela entra com `exog=` no ajuste e na
previsão. Um exemplo: se a nossa série tivesse um pico em todo feriado, uma
coluna 0/1 de feriados seria a exógena.

No `statsmodels`, o SARIMAX é uma **regressão com erros SARIMA**: $y_t = \beta^\top x_t + \eta_t$, em que
$\eta_t$ segue um SARIMA. A exógena precisa ser conhecida também no futuro — a previsão do tempo para
amanhã serve; a temperatura de amanhã, não.

Não vamos usar exógenas hoje. Fica para quem quiser no projeto.

---
## 9. Para ir além

**A metodologia de Box e Jenkins** organiza tudo o que fizemos num ciclo: *identificação*
(estacionariedade, ACF e PACF), *estimação* (máxima verossimilhança), *diagnóstico* (resíduos de
ruído branco) — e volta ao começo se o diagnóstico falhar.

**KPSS, o teste complementar.** O ADF tem como hipótese nula a raiz unitária; o KPSS
(`statsmodels.tsa.stattools.kpss`) tem como nula a estacionariedade. Usar os dois evita concluir
"estacionária" só porque o ADF não teve poder para rejeitar.

**A incerteza da previsão cresce com o horizonte — quando há raiz unitária.** Num passeio aleatório,
o erro de previsão $h$ passos à frente é a soma de $h$ choques: variância $h\sigma^2$, e o intervalo
alarga como $\sqrt{h}$. No nosso SARIMA, com $\theta \approx \Theta \approx -1$, as diferenças se cancelam
e o intervalo quase não alarga:

In [ ]:
fc_sarima = ajuste.get_forecast(40).conf_int(alpha=0.05)
fc_passeio = SARIMAX(treino.values, order=(0, 1, 0), trend="t").fit(disp=False).get_forecast(40).conf_int(alpha=0.05)
for nome, ic in [("SARIMA(0,1,1)(0,1,1,7)", fc_sarima), ("passeio com deriva, ARIMA(0,1,0)", fc_passeio)]:
    print(f"{nome:34s} largura do IC 95%:  h = 1 → {ic[0, 1] - ic[0, 0]:5.1f}   h = 40 → {ic[-1, 1] - ic[-1, 0]:5.1f}")
razao = (fc_passeio[-1, 1] - fc_passeio[-1, 0]) / (fc_passeio[0, 1] - fc_passeio[0, 0])
print(f"no passeio, a largura cresceu {razao:.2f} vezes; √40 = {np.sqrt(40):.2f}")

O intervalo de 95% deveria conter 95% dos pontos. Conferimos nos 40 dias de teste, previstos de uma
vez a partir do fim do treino:

In [ ]:
def dentro(ic):
    return int(np.sum((teste.values >= ic[:, 0]) & (teste.values <= ic[:, 1])))

print(f"pontos do teste dentro do IC 95%:  SARIMA {dentro(fc_sarima)} de 40   passeio {dentro(fc_passeio)} de 40")
print(f"σ estimado pelo SARIMA no treino = {np.sqrt(ajuste.params[-1]):.2f}   (o ruído foi gerado com σ = 4)")
print(f"desvio padrão das diferenças diárias no treino = {treino.diff().std():.1f}")

**Discussão.**
* **SARIMA:** a largura vai de 14,1 em $h = 1$ para 14,2 em $h = 40$ — não cresce. Com
  $\theta \approx \Theta \approx -1$, o modelo equivale a "tendência e sazonalidade fixas mais ruído": o
  erro de amanhã e o de daqui a 40 dias são o mesmo ruído. A largura é cerca de 4 vezes o $\hat\sigma$.
* **Passeio com deriva:** de 39,5 para 249,8, exatamente $\sqrt{40}$ vezes. Ele trata a oscilação
  semanal como choques aleatórios: o $\sigma$ que ele estima é o desvio das diferenças diárias, 10,1,
  e não o ruído de 4.
* **Cobertura:** o passeio cobre 39 dos 40 pontos — mas com um intervalo de 250 de largura numa
  série que, no teste, fica entre 210 e 275 (veja o gráfico das previsões). Acerta sem informar
  nada. O SARIMA cobre 35 de 40, menos que os 95% (38) prometidos: o $\hat\sigma$ do treino saiu 3,48, abaixo do 4 com que o ruído foi
  gerado. O intervalo é tão bom quanto a estimativa de $\sigma$ e a hipótese de que o futuro se
  comporta como o passado.

**A lição:** escolher o modelo muda também a incerteza. Um intervalo largo demais é tão inútil quanto
um estreito demais; o que se quer é o mais estreito que ainda cubra o que promete.

### Referências
* Hyndman, R. J.; Athanasopoulos, G. *Forecasting: principles and practice*, 3ª ed. OTexts, 2021.
  Gratuito em https://otexts.com/fpp3/ — capítulos de ARIMA e de avaliação de previsões.
* Ferreira, P. C. *Análise de Séries Temporais*, 1ª ed. Elsevier, 2017 — em português, na pasta da
  disciplina.
* Box, G. E. P.; Jenkins, G. M. *Time Series Analysis: Forecasting and Control*. Holden-Day, 1970.
* Dickey, D. A.; Fuller, W. A. Distribution of the estimators for autoregressive time series with a
  unit root. *Journal of the American Statistical Association*, 74(366), 1979.
* Kwiatkowski, D. e colegas. Testing the null hypothesis of stationarity against the alternative of a
  unit root. *Journal of Econometrics*, 54, 1992.
* Ljung, G. M.; Box, G. E. P. On a measure of lack of fit in time series models. *Biometrika*, 65(2), 1978.
* Hyndman, R. J.; Koehler, A. B. Another look at measures of forecast accuracy. *International Journal
  of Forecasting*, 22(4), 2006.
* Tashman, L. J. Out-of-sample tests of forecasting accuracy: an analysis and review. *International
  Journal of Forecasting*, 16(4), 2000.

---
# Atividade ST

**Série:** `data/AirPassengers.csv` — passageiros de companhias aéreas por mês,
de 1949 a 1960.

**Faça, num notebook seu:**

1. Plote a série. Ela é aditiva ou multiplicativa (Aula 01)? Se for multiplicativa,
   trabalhe com o logaritmo e **volte à escala original** antes de calcular as métricas.
2. Aplique o ADF à série original, ao log e às diferenças que você escolher.
   Justifique `d` e `D`.
3. Plote ACF e PACF da série estacionária. Proponha `(p,d,q)(P,D,Q,m)` e **escreva
   por que**, citando os lags que você viu.
4. Separe os **últimos 24 meses** como teste. Faça *walk-forward* de um passo.
5. Compare, com MAE, RMSE e MASE (`m = 12`):
   ingênua, ingênua sazonal, um ARIMA não sazonal e o seu SARIMA.
6. Valide o seu melhor modelo com janela em expansão
   (`TimeSeriesSplit(n_splits=3, test_size=12)`). Informe o MASE médio e o desvio padrão.
7. Conclusão em até cinco linhas: qual modelo, por quê, e quanto ele ganha da
   ingênua sazonal.

**Cuidados que fazem diferença:**
* Métricas sempre na **escala original** — a do log engana.
* O teste **nunca** entra no ajuste: cada previsão usa só o passado (vazamento).
* Ordens justificadas pelos lags que você viu na ACF e na PACF, não por tentativa.
* Toda comparação inclui as linhas de base: um modelo que não ganha da ingênua sazonal não aprendeu nada.